# Can we predict term-deposit subscription (`y`) after a marketing call?

**Plan:** explore the data, build a baseline and a gradient-boosting model, evaluate with metrics suited to an imbalanced target (ROC-AUC, PR-AUC, lift), and check the leakage risk of `duration` (call length is only known *after* the call, so it can't be used to decide whom to call).

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve, classification_report
from sklearn.inspection import permutation_importance
RS=42
df=pd.read_csv('data.csv')
df.shape

## 1. Data overview

In [ ]:
display(df.head())
display(df.isna().sum().to_frame('missing').assign(pct=lambda d:(d.missing/len(df)*100).round(1)).query('missing>0'))
print('duplicates:',df.duplicated().sum())
df['y'].value_counts(normalize=True)

In [ ]:
df['target']=(df['y']=='yes').astype(int)
num=['age','balance','day_of_week','duration','campaign','pdays','previous']
display(df[num].describe().T)
print(df['pdays'].eq(-1).mean(), 'share with pdays=-1 (not previously contacted)')

## 2. Exploration: subscription rate by feature

In [ ]:
cat=[c for c in df.columns if df[c].dtype==object and c!='y']
fig,axs=plt.subplots(3,4,figsize=(18,11)); axs=axs.ravel()
for ax,col in zip(axs,cat):
    r=df.fillna({col:'(missing)'}).groupby(col)['target'].agg(['mean','size'])
    r['mean'].sort_values().plot.barh(ax=ax,color='#3b7dd8'); ax.axvline(df.target.mean(),color='k',ls='--',lw=1)
    ax.set_title(col); ax.set_xlabel('subscription rate')
for ax in axs[len(cat):]: ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
fig,axs=plt.subplots(1,3,figsize=(16,4))
for ax,col in zip(axs,['age','balance','duration']):
    d=df[[col,'target']].copy(); d['bin']=pd.qcut(d[col],10,duplicates='drop')
    d.groupby('bin',observed=True)['target'].mean().plot(marker='o',ax=ax); ax.set_title(f'rate by {col} decile'); ax.tick_params(axis='x',rotation=60)
plt.tight_layout(); plt.show()

## 3. Modelling setup
`duration` is known only after the call ends, so a model using it answers "who subscribed given how the call went", not "whom should we call". I fit two feature sets:
- **A – deployable**: everything known before the call (no `duration`)
- **B – with `duration`**: an upper bound, for reference only

Stratified 80/20 hold-out, plus 5-fold CV on the training part. Missing values are imputed (categoricals get an explicit `missing` category).

In [ ]:
X=df.drop(columns=['y','target']); yv=df['target']
Xtr,Xte,ytr,yte=train_test_split(X,yv,test_size=.2,stratify=yv,random_state=RS)
def prep(cols):
    nc=[c for c in cols if c in num]; cc=[c for c in cols if c not in num]
    return ColumnTransformer([('n',Pipeline([('i',SimpleImputer(strategy='median')),('s',StandardScaler())]),nc),
      ('c',Pipeline([('i',SimpleImputer(strategy='constant',fill_value='missing')),('o',OneHotEncoder(handle_unknown='ignore'))]),cc)])
def models(cols):
    return {'Dummy':Pipeline([('p',prep(cols)),('m',DummyClassifier(strategy='prior'))]),
     'Logistic':Pipeline([('p',prep(cols)),('m',LogisticRegression(max_iter=2000,class_weight='balanced'))]),
     'HistGB':Pipeline([('p',prep(cols)),('m',HistGradientBoostingClassifier(learning_rate=.05,max_iter=300,l2_regularization=1,random_state=RS))])}
sets={'A: no duration':[c for c in X.columns if c!='duration'],'B: with duration':list(X.columns)}
cv=StratifiedKFold(5,shuffle=True,random_state=RS)
rows=[];fitted={}
for sn,cols in sets.items():
    for mn,m in models(cols).items():
        cvauc=cross_val_score(m,Xtr[cols],ytr,cv=cv,scoring='roc_auc').mean()
        m.fit(Xtr[cols],ytr); p=m.predict_proba(Xte[cols])[:,1]; fitted[(sn,mn)]=p
        rows.append(dict(features=sn,model=mn,cv_auc=cvauc,test_auc=roc_auc_score(yte,p),test_pr_auc=average_precision_score(yte,p)))
res=pd.DataFrame(rows).round(3); res

In [ ]:
print('baseline positive rate:',round(yte.mean(),3))
fig,axs=plt.subplots(1,2,figsize=(12,4.5))
for (sn,mn),p in fitted.items():
    if mn=='Dummy': continue
    ls='-' if sn.startswith('A') else '--'
    f,t,_=roc_curve(yte,p); axs[0].plot(f,t,ls,label=f'{sn} / {mn}')
    pr,rc,_=precision_recall_curve(yte,p); axs[1].plot(rc,pr,ls,label=f'{sn} / {mn}')
axs[0].plot([0,1],[0,1],'k:');axs[0].set(title='ROC',xlabel='FPR',ylabel='TPR')
axs[1].axhline(yte.mean(),color='k',ls=':');axs[1].set(title='Precision-Recall',xlabel='recall',ylabel='precision')
axs[0].legend(fontsize=8); plt.tight_layout(); plt.show()

## 4. Business view: lift for the deployable model (A, HistGB)
If we only call the top-scored clients, what share of subscribers do we capture?

In [ ]:
p=fitted[('A: no duration','HistGB')]
d=pd.DataFrame({'p':p,'y':yte.values}).sort_values('p',ascending=False).reset_index(drop=True)
d['decile']=pd.qcut(d.index,10,labels=range(1,11))
g=d.groupby('decile',observed=True)['y'].agg(['mean','sum']).rename(columns={'mean':'rate','sum':'subs'})
g['lift']=g.rate/d.y.mean(); g['cum_capture']=g.subs.cumsum()/d.y.sum()
display(g.round(3))
g['rate'].plot.bar(color='#3b7dd8'); plt.axhline(d.y.mean(),color='k',ls='--'); plt.ylabel('subscription rate'); plt.title('Rate by score decile (1 = highest)'); plt.show()

In [ ]:
thr=np.quantile(p,.8)  # contact top 20%
print(classification_report(yte,(p>=thr).astype(int),target_names=['no','yes']))

## 5. What drives the predictions? (permutation importance, model A)

In [ ]:
mA=models(sets['A: no duration'])['HistGB'].fit(Xtr[sets['A: no duration']],ytr)
pi=permutation_importance(mA,Xte[sets['A: no duration']],yte,scoring='roc_auc',n_repeats=5,random_state=RS)
imp=pd.Series(pi.importances_mean,index=sets['A: no duration']).sort_values()
imp.plot.barh(xerr=pd.Series(pi.importances_std,index=imp.index)[imp.index],color='#3b7dd8'); plt.xlabel('drop in test ROC-AUC'); plt.show()

## Conclusions

In [ ]:
a=res.query("features=='A: no duration' and model=='HistGB'").iloc[0]; b=res.query("features=='B: with duration' and model=='HistGB'").iloc[0]
print(f"Deployable model (no duration): test ROC-AUC {a.test_auc:.3f}, PR-AUC {a.test_pr_auc:.3f} (base rate {yte.mean():.3f})")
print(f"With duration (leaky ref.)    : test ROC-AUC {b.test_auc:.3f}, PR-AUC {b.test_pr_auc:.3f}")
print(f"Top decile captures {g.cum_capture.iloc[0]:.0%} of subscribers; top 3 deciles {g.cum_capture.iloc[2]:.0%}.")

**Caveats**
- `duration` is a post-call outcome; B overstates what is achievable when choosing whom to call. Use A for targeting; B is only useful for in-call/after-call analysis.
- Heavy class imbalance: accuracy is misleading; judge by AUC/PR-AUC/lift.
- Single random split of one dataset; no temporal validation (`month`/`day_of_week` hint at time ordering, so real-world performance on future campaigns may be lower). `poutcome`/`contact` have many missing values; the missingness itself may be informative.
- Models are lightly configured, not tuned.